In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("ETL Task")
         .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
         .getOrCreate())

jdbc_url = "jdbc:postgresql://localhost:5432/etl_pipeline"
props = {
    "user": "postgres",
    "password": "123456",
    "driver": "org.postgresql.Driver"
}

movies = spark.read.jdbc(jdbc_url, "movies", properties=props)
users = spark.read.jdbc(jdbc_url, "users", properties=props) 
profiles = spark.read.jdbc(jdbc_url, "user_profiles", properties=props)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/02 18:39:20 WARN Utils: Your hostname, DoubleCats.local, resolves to a loopback address: 127.0.0.1; using 192.168.100.44 instead (on interface en0)
26/03/02 18:39:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/opt/anaconda3/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/dzhen973/.ivy2.5.2/cache
The jars for the packages stored in: /Users/dzhen973/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ece4418e-1288-45fd-b1b2-f991109edf75;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
downloading https://repo1.maven.org/maven2/org/postgresql/postgresql/42.7.3/postgresql-42.7.3.jar ...
	[SUCCESSFU

In [3]:
movies.show()

+---+--------------------+--------------------+--------+
| id|                name|         description|category|
+---+--------------------+--------------------+--------+
|  1|              Avatar|Avatar is a 2009 ...|  Sci-Fi|
|  2|Avengers: Infinit...|Avengers: Infinit...|  Sci-Fi|
|  3|            Holidate|Holidate is a 202...|  Romcom|
|  4|          Extraction|Extraction is a 2...|  Action|
|  5|           John Wick|John Wick is a 20...|  Action|
+---+--------------------+--------------------+--------+



## Task 1

In [23]:
df = (users
      .join(profiles, users["id"] == profiles["user_id"], "inner")
      .join(movies, users["movie_id"] == movies["id"], "inner")
)
df.orderBy(users["id"]).show()

+---+--------+------+-------+---+------+----------+---+--------------------+--------------------+--------+
| id|movie_id|rating|user_id|age|gender|occupation| id|                name|         description|category|
+---+--------+------+-------+---+------+----------+---+--------------------+--------------------+--------+
|  1|       1|     4|      1| 25|     M|  Engineer|  1|              Avatar|Avatar is a 2009 ...|  Sci-Fi|
|  2|       2|     5|      2| 34|     F|  Designer|  2|Avengers: Infinit...|Avengers: Infinit...|  Sci-Fi|
|  3|       1|     4|      3| 19|     M|   Student|  1|              Avatar|Avatar is a 2009 ...|  Sci-Fi|
|  4|       3|     3|      4| 45|     F|   Manager|  3|            Holidate|Holidate is a 202...|  Romcom|
|  5|       4|     5|      5| 22|     M|   Student|  4|          Extraction|Extraction is a 2...|  Action|
+---+--------+------+-------+---+------+----------+---+--------------------+--------------------+--------+



In [25]:
df = df.withColumn(
    "age_group",
    F.when(F.col("age") < 18, "0-17")
     .when((F.col("age") >= 18) & (F.col("age") <= 25), "18-25")
     .when((F.col("age") >= 26) & (F.col("age") <= 35), "26-35")
     .when((F.col("age") >= 36) & (F.col("age") <= 45), "36-45")
     .otherwise("46+")
)
df.orderBy(users["id"]).show()

+---+--------+------+-------+---+------+----------+---+--------------------+--------------------+--------+---------+
| id|movie_id|rating|user_id|age|gender|occupation| id|                name|         description|category|age_group|
+---+--------+------+-------+---+------+----------+---+--------------------+--------------------+--------+---------+
|  1|       1|     4|      1| 25|     M|  Engineer|  1|              Avatar|Avatar is a 2009 ...|  Sci-Fi|    18-25|
|  2|       2|     5|      2| 34|     F|  Designer|  2|Avengers: Infinit...|Avengers: Infinit...|  Sci-Fi|    26-35|
|  3|       1|     4|      3| 19|     M|   Student|  1|              Avatar|Avatar is a 2009 ...|  Sci-Fi|    18-25|
|  4|       3|     3|      4| 45|     F|   Manager|  3|            Holidate|Holidate is a 202...|  Romcom|    36-45|
|  5|       4|     5|      5| 22|     M|   Student|  4|          Extraction|Extraction is a 2...|  Action|    18-25|
+---+--------+------+-------+---+------+----------+---+---------

In [19]:
avg_df = (df.groupBy("category", "age_group")  # 按照category，分组
            .agg(F.avg("rating").alias("avg_rating")) # 新增avg_rating列，求每个组的rating平均值
         )

avg_df.orderBy("category", F.desc("avg_rating")).show()

+--------+---------+----------+
|category|age_group|avg_rating|
+--------+---------+----------+
|  Action|    18-25|       5.0|
|  Romcom|    36-45|       3.0|
|  Sci-Fi|    26-35|       5.0|
|  Sci-Fi|    18-25|       4.0|
+--------+---------+----------+



In [21]:
w = Window.partitionBy("category") # 把数据按 category 分成多个组（partitio），每个类别一组
           .orderBy(F.desc("avg_rating")) # 在每个类别组内，按平均分从高到低排序

best_df = (avg_df
           .withColumn("rn", F.row_number().over(w))  # 给每行加一个名次 rn
           .filter(F.col("rn") == 1) # 只保留每个类别的第一名（最高平均分的年龄段）
           .drop("rn") # 把辅助列删掉，输出干净
           .orderBy("category")
          )

best_df.show()

+--------+---------+----------+
|category|age_group|avg_rating|
+--------+---------+----------+
|  Action|    18-25|       5.0|
|  Romcom|    36-45|       3.0|
|  Sci-Fi|    26-35|       5.0|
+--------+---------+----------+



In [27]:
output_table = "agegroup_by_category_output"

(best_df.write
   .mode("overwrite") 
   .jdbc(jdbc_url, output_table, properties=props)
)

In [29]:
agegroup_by_category_output = spark.read.jdbc(jdbc_url, "agegroup_by_category_output", properties=props)
agegroup_by_category_output.show()

+--------+---------+----------+
|category|age_group|avg_rating|
+--------+---------+----------+
|  Action|    18-25|       5.0|
|  Romcom|    36-45|       3.0|
|  Sci-Fi|    26-35|       5.0|
+--------+---------+----------+



## Task 2

In [33]:
promotions = spark.read.jdbc(jdbc_url, "promotions", properties=props)
promotions.show()

+--------+--------------------+
|movie_id|              budget|
+--------+--------------------+
|       1|100000.0000000000...|
|       2|300000.0000000000...|
|       3|50000.00000000000...|
|       4|200000.0000000000...|
|       5|150000.0000000000...|
+--------+--------------------+



In [49]:
avg_ratings = (users
               .groupBy("movie_id")
               .agg(F.avg("rating").alias("avg_rating")))
avg_ratings.orderBy(avg_ratings["movie_id"]).show()

+--------+----------+
|movie_id|avg_rating|
+--------+----------+
|       1|       4.0|
|       2|       5.0|
|       3|       3.0|
|       4|       5.0|
+--------+----------+



In [53]:
joined = (promotions
          .join(avg_ratings, on="movie_id", how="left"))
joined.orderBy(joined["movie_id"]).show()

+--------+--------------------+----------+
|movie_id|              budget|avg_rating|
+--------+--------------------+----------+
|       1|100000.0000000000...|       4.0|
|       2|300000.0000000000...|       5.0|
|       3|50000.00000000000...|       3.0|
|       4|200000.0000000000...|       5.0|
|       5|150000.0000000000...|      NULL|
+--------+--------------------+----------+



In [63]:
result = (joined
          .withColumn("avg_rating", F.coalesce(F.col("avg_rating"), F.lit(0.0))) # 如果 avg_rating 是 null，就用 0 替代
          .withColumn(
              "efficiency",
              F.when(F.col("budget") > 0,   # 防止发生除0异常
                     F.col("avg_rating") / (F.col("budget") / F.lit(10000.0)))
               .otherwise(F.lit(0.0))
          ))
result.orderBy(result["movie_id"]).show()

+--------+--------------------+----------+-------------------+
|movie_id|              budget|avg_rating|         efficiency|
+--------+--------------------+----------+-------------------+
|       1|100000.0000000000...|       4.0|                0.4|
|       2|300000.0000000000...|       5.0|0.16666666666666666|
|       3|50000.00000000000...|       3.0|                0.6|
|       4|200000.0000000000...|       5.0|               0.25|
|       5|150000.0000000000...|       0.0|                0.0|
+--------+--------------------+----------+-------------------+



In [67]:
# join进name和category两个字段，单纯方便阅读
result_with_name = (result
                .join(movies.select(F.col("id").alias("movie_id"),
                                    F.col("name"),
                                    F.col("category")),
                      on="movie_id", how="left"))
result_with_name.orderBy(result_named["movie_id"]).show()

+--------+--------------------+----------+-------------------+--------------------+--------+
|movie_id|              budget|avg_rating|         efficiency|                name|category|
+--------+--------------------+----------+-------------------+--------------------+--------+
|       1|100000.0000000000...|       4.0|                0.4|              Avatar|  Sci-Fi|
|       2|300000.0000000000...|       5.0|0.16666666666666666|Avengers: Infinit...|  Sci-Fi|
|       3|50000.00000000000...|       3.0|                0.6|            Holidate|  Romcom|
|       4|200000.0000000000...|       5.0|               0.25|          Extraction|  Action|
|       5|150000.0000000000...|       0.0|                0.0|           John Wick|  Action|
+--------+--------------------+----------+-------------------+--------------------+--------+



In [73]:
final_df = (result_with_name
            .select("movie_id", "name", "category", "budget", "avg_rating", "efficiency")
            .orderBy(F.desc("efficiency")))

final_df.orderBy(final_df["movie_id"]).show()

+--------+--------------------+--------+--------------------+----------+-------------------+
|movie_id|                name|category|              budget|avg_rating|         efficiency|
+--------+--------------------+--------+--------------------+----------+-------------------+
|       1|              Avatar|  Sci-Fi|100000.0000000000...|       4.0|                0.4|
|       2|Avengers: Infinit...|  Sci-Fi|300000.0000000000...|       5.0|0.16666666666666666|
|       3|            Holidate|  Romcom|50000.00000000000...|       3.0|                0.6|
|       4|          Extraction|  Action|200000.0000000000...|       5.0|               0.25|
|       5|           John Wick|  Action|150000.0000000000...|       0.0|                0.0|
+--------+--------------------+--------+--------------------+----------+-------------------+

